# Session 1: Amazon Bedrock을 통한 Foundation Model 호출

### Index
1. Amazon Bedrock 소개
2. 사용 가능한 Model 조회
3. CLI를 통한 모델 호출
   - invoke-model vs converse 비교
4. SDK(boto3)를 통한 모델 호출
   - invoke_model
   - converse
   - Streaming (invoke_model_with_response_stream / converse_stream)
5. System Prompt & inferenceConfig

---
## Amazon Bedrock이란?

Amazon Bedrock은 AWS의 **완전 관리형 생성 AI 서비스**입니다.  
다양한 Foundation Model(FM)을 **API 하나로** 호출할 수 있으며, 인프라 관리 없이 바로 사용할 수 있습니다.

### 주요 특징
- **다양한 모델 선택**: Anthropic Claude, Amazon Nova, Meta Llama, Mistral 등
- **서버리스**: 프로비저닝 없이 API 호출만으로 사용
- **보안**: 데이터가 모델 학습에 사용되지 않음
- **통합**: AWS 서비스(IAM, CloudWatch, Lambda 등)와 자연스러운 연동

### 호출 방식 2가지

| 방식 | 설명 | 특징 |
|------|------|------|
| `invoke-model` / `invoke_model` | 모델별 고유 body 포맷 사용 | 모델마다 요청/응답 구조가 다름 |
| `converse` | 모델 무관 통일 인터페이스 | 모델을 바꿔도 코드 변경 불필요 **(권장)** |

> 💡 **왜 Converse가 권장인가?**  
> `invoke_model`은 Claude는 `anthropic_version` 필드가 필요하고, Nova는 `inferenceConfig` 구조가 다릅니다.  
> 모델을 교체할 때마다 body를 수정해야 합니다.  
> `converse`는 동일한 메시지 포맷으로 어떤 모델이든 호출할 수 있어 유지보수가 쉽습니다.

## 사용 가능한 Model 리스트 조회

In [1]:
! aws bedrock list-foundation-models --region us-east-1 \
  --query 'modelSummaries[].[modelId, modelName, inferenceTypesSupported[0]]' \
  --output table

---------------------------------------------------------------------------------------------------------------
|                                            ListFoundationModels                                             |
+------------------------------------------------+---------------------------------------+--------------------+
|  nvidia.nemotron-nano-12b-v2                   |  NVIDIA Nemotron Nano 12B v2 VL BF16  |  ON_DEMAND         |
|  qwen.qwen3-coder-next                         |  Qwen3 Coder Next                     |  ON_DEMAND         |
|  anthropic.claude-sonnet-4-20250514-v1:0       |  Claude Sonnet 4                      |  INFERENCE_PROFILE |
|  anthropic.claude-haiku-4-5-20251001-v1:0      |  Claude Haiku 4.5                     |  INFERENCE_PROFILE |
|  moonshotai.kimi-k2.5                          |  Kimi K2.5                            |  ON_DEMAND         |
|  openai.gpt-oss-120b-1:0                       |  gpt-oss-120b                         |  ON_DEMAND   

## CLI

> AWS CLI의 `bedrock-runtime` 서브커맨드를 사용합니다.  
> 아래에서 같은 "안녕" 메시지를 `invoke-model`과 `converse`로 각각 보내며 **body 포맷 차이**를 비교해보세요.

### Invoke Model

> ⚠️ 모델마다 body 포맷이 다릅니다.  
> - Nova: `messages[].content[].text` 구조 + `inferenceConfig`  
> - Claude: `messages[].content` (문자열) + `anthropic_version` 필수

In [6]:
# Nova Lite 2
! aws bedrock-runtime invoke-model \
    --region us-east-1 \
    --model-id "us.amazon.nova-2-lite-v1:0" \
    --body '{"messages":[{"role":"user","content":[{"text":"안녕"}]}],"inferenceConfig":{"max_new_tokens":512}}' \
    --cli-binary-format raw-in-base64-out \
    /dev/stdout | jq .

{
  "output": {
    "message": {
      "content": [
        {
          "text": "안녕하세요! 어떻게 도와드릴까요?"
        }
      ],
      "role": "assistant"
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 49,
    "outputTokens": 16,
    "totalTokens": 65,
    "cacheReadInputTokenCount": 0,
    "cacheWriteInputTokenCount": 0
  }
}
{
  "contentType": "application/json"
}


In [4]:
# Claude Sonnet 4.5
! aws bedrock-runtime invoke-model \
    --region us-east-1 \
    --model-id us.anthropic.claude-sonnet-4-5-20250929-v1:0 \
    --body '{"anthropic_version":"bedrock-2023-05-31","max_tokens":1024,"messages":[{"role":"user","content":"안녕"}]}' \
    --cli-binary-format raw-in-base64-out \
    /dev/stdout | jq .

{
  "model": "claude-sonnet-4-5-20250929",
  "id": "msg_bdrk_01N4w7EC5MNMV96N3LH9m6N1",
  "type": "message",
  "role": "assistant",
  "content": [
    {
      "type": "text",
      "text": "안녕하세요! 만나서 반갑습니다. 무엇을 도와드릴까요?"
    }
  ],
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "usage": {
    "input_tokens": 11,
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "cache_creation": {
      "ephemeral_5m_input_tokens": 0,
      "ephemeral_1h_input_tokens": 0
    },
    "output_tokens": 37
  }
}
{
  "contentType": "application/json"
}


### Converse

> ✅ 모델에 관계없이 동일한 메시지 포맷을 사용합니다.  
> `modelId`만 바꾸면 다른 모델로 전환 가능 — 코드 수정 불필요!

In [8]:
# Nova Lite 2
! aws bedrock-runtime converse \
    --region us-east-1 \
    --model-id us.amazon.nova-2-lite-v1:0 \
    --messages '[{"role":"user","content":[{"text":"안녕"}]}]'

{
    "output": {
        "message": {
            "role": "assistant",
            "content": [
                {
                    "text": "안녕하세요! 어떻게 도와드릴까요?"
                }
            ]
        }
    },
    "stopReason": "end_turn",
    "usage": {
        "inputTokens": 49,
        "outputTokens": 16,
        "totalTokens": 65
    },
    "metrics": {
        "latencyMs": 470
    }
}


In [9]:
# Claude Sonnet 4.5
! aws bedrock-runtime converse \
    --region us-east-1 \
    --model-id "us.anthropic.claude-sonnet-4-5-20250929-v1:0" \
    --messages '[{"role":"user","content":[{"text":"안녕"}]}]'

{
    "output": {
        "message": {
            "role": "assistant",
            "content": [
                {
                    "text": "안녕하세요! 반갑습니다. 무엇을 도와드릴까요? 😊"
                }
            ]
        }
    },
    "stopReason": "end_turn",
    "usage": {
        "inputTokens": 11,
        "outputTokens": 36,
        "totalTokens": 47,
        "cacheReadInputTokens": 0,
        "cacheWriteInputTokens": 0
    },
    "metrics": {
        "latencyMs": 4315
    }
}


## SDK(boto3)

> Python에서 `boto3` 클라이언트를 사용하여 Bedrock 모델을 호출합니다.  
> CLI와 동일하게 `invoke_model`(모델별 포맷)과 `converse`(통일 포맷) 두 가지 방식이 있습니다.

In [6]:
# ! python3.14 -m pip install boto3

In [13]:
import boto3
import json

In [14]:
client = boto3.client("bedrock-runtime", region_name="us-east-1")

### Invoke

> 모델별 고유 body를 JSON으로 직접 구성합니다.  
> 응답 파싱도 모델마다 다르다는 점에 주목하세요. (Claude: `content[0].text`, Nova: `output.message.content[0].text`)

In [9]:
# Claude Sonnet 4.5
response = client.invoke_model(
  modelId="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
  body=json.dumps({
      "anthropic_version": "bedrock-2023-05-31",
      "max_tokens": 1024,
      "messages": [{"role": "user", "content": "안녕"}]
  })
)
print(json.loads(response["body"].read())["content"][0]["text"])

안녕하세요! 반갑습니다. 무엇을 도와드릴까요?


In [10]:
# Nova 2 Lite
response = client.invoke_model(
  modelId="us.amazon.nova-2-lite-v1:0",
  body=json.dumps({
      "messages": [{"role": "user", "content": [{"text": "안녕"}]}],
      "inferenceConfig": {"maxTokens": 512}
  })
)
print(json.loads(response["body"].read())["output"]["message"]["content"][0]["text"])

안녕하세요! 어떻게 도와드릴까요?


### Converse

> ✅ 동일한 `messages` 포맷, 동일한 응답 구조.  
> `modelId`만 교체하면 됩니다. 프로덕션에서는 이 방식을 권장합니다.

In [11]:
messages = [{"role": "user", "content": [{"text": "안녕"}]}]
nova_modelId = "us.amazon.nova-2-lite-v1:0"
claude_modelId = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

In [15]:
# Claude Sonnet 4.5
response = client.converse(
  modelId=claude_modelId,
  messages=messages
)
print(response["output"]["message"]["content"][0]["text"])

안녕하세요! 어떻게 도와드릴까요?


In [16]:
# Nova 2 Lite
response = client.converse(
  modelId=nova_modelId,
  messages=messages
)
print(response["output"]["message"]["content"][0]["text"])

안녕하세요! 어떻게 도와드릴까요?


### Stream

> 긴 응답을 생성할 때 전체 완료를 기다리지 않고 **토큰 단위로 실시간 수신**합니다.  
> 챗봇 UX에서 필수적인 패턴입니다.
>
> | 방식 | API | 비고 |
> |------|-----|------|
> | invoke_model_with_response_stream | 모델별 body 포맷 | 모델별 스트림 파싱 다름 |
> | converse_stream | 통일 포맷 | 권장 |

In [ ]:
# ============================================================
# invoke_model_with_response_stream
# ============================================================
import boto3
import json

client = boto3.client("bedrock-runtime", region_name="us-east-1")

# Claude Sonnet 4.5
response = client.invoke_model_with_response_stream(
  modelId="us.amazon.nova-2-lite-v1:0",
  body=json.dumps({
      "anthropic_version": "bedrock-2023-05-31",
      "max_tokens": 1024,
      "messages": [{"role": "user", "content": "안녕. 오늘 날씨 알려주고, AWS Bedrock의 서비스들에 대해서 설명해줘."}]
  })
)

for event in response["body"]:
  chunk = json.loads(event["chunk"]["bytes"])
  if chunk.get("type") == "content_block_delta":
      print(chunk["delta"]["text"], end="", flush=True)

# 안녕하세요! 👋

## 오늘 날씨에 대해

죄송하지만, 저는 실시간 정보에 접근할 수 없어서 현재 날씨를 알려드릴 수 없습니다. 날씨 정보는 다음을 이용해주세요:
- 기상청 날씨누리 (weather.go.kr)
- 네이버/구글 날씨 검색
- 날씨 앱

---

## AWS Bedrock 서비스 설명

AWS Bedrock은 **완전 관리형 생성 AI 서비스**로, 다양한 기초 모델(Foundation Models)을 API를 통해 사용할 수 있게 해주는 플랫폼입니다.

### 주요 특징

**1. 다양한 AI 모델 제공**
- **Anthropic Claude** - 대화형 AI, 긴 컨텍스트 처리
- **Amazon Titan** - 텍스트/임베딩 생성
- **AI21 Labs Jurassic** - 텍스트 생성
- **Stability AI** - 이미지 생성
- **Cohere** - 텍스트 생성 및 임베딩
- **Meta Llama** - 오픈소스 기반 모델

**2. 주요 기능**
- **모델 커스터마이징**: 자체 데이터로 파인튜닝 가능
- **RAG (검색 증강 생성)**: 지식 베이스 연동
- **에이전트**: 자동화된 작업 수행
- **보안 & 프라이버시**: 데이터가 학습에 사용되지 않음
- **서버리스**: 인프라 관리 불필요

**3. 활용 사례**
- 챗봇 및 가상 비서
- 콘텐츠 생성 및 요약
- 문서 분석 및 검색
- 코드 생성
- 이미지 생성

도움이 되었나요? 특정 부분에 대해 더 자세히 알고 싶으신가요?

In [ ]:
# ============================================================
# invoke_model
# ============================================================

import boto3
import json

client = boto3.client("bedrock-runtime", region_name="us-east-1")

# Claude Sonnet 4.5
response = client.invoke_model(
  modelId="us.amazon.nova-2-lite-v1:0",
  body=json.dumps({
      "anthropic_version": "bedrock-2023-05-31",
      "max_tokens": 1024,
      "messages": [{"role": "user", "content": "안녕. 오늘 날씨 알려주고, AWS Bedrock의 서비스들에 대해서 설명해줘."}]
  })
)
print(json.loads(response["body"].read())["content"][0]["text"])

# 안녕하세요! 👋

## 날씨 정보 ☁️
죄송하지만, 저는 실시간 날씨 정보에 접근할 수 없습니다. 현재 날씨를 확인하시려면:
- 포털 사이트 (네이버, 다음 등)
- 기상청 날씨누리
- 날씨 앱 사용을 권장드립니다

## AWS Bedrock 서비스 설명 🤖

**AWS Bedrock**은 Amazon의 완전 관리형 생성 AI 서비스입니다.

### 주요 특징:

1. **Foundation Models (기반 모델) 제공**
   - Anthropic Claude
   - AI21 Labs Jurassic
   - Stability AI Stable Diffusion
   - Amazon Titan
   - Meta Llama
   - Cohere Command

2. **핵심 기능**
   - 🔧 **모델 커스터마이징**: 자체 데이터로 파인튜닝 가능
   - 🔒 **보안**: 데이터가 AWS 인프라 내에서 안전하게 처리
   - 📊 **통합**: 다른 AWS 서비스와 쉬운 연동
   - 💰 **종량제 요금**: 사용한 만큼만 지불

3. **주요 사용 사례**
   - 챗봇 및 가상 비서
   - 콘텐츠 생성 및 요약
   - 이미지 생성
   - 코드 생성
   - 검색 및 추천 시스템

궁금한 점이 더 있으시면 말씀해주세요!


### converse_stream (권장)

> ✅ 통일된 포맷으로 스트리밍. 모델 교체 시에도 파싱 코드 변경 불필요.

In [27]:
# ============================================================
# converse_stream (권장)
# ============================================================

response = client.converse_stream(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS Bedrock의 주요 기능을 3가지만 설명해줘."}]}]
)

for event in response["stream"]:
  if "contentBlockDelta" in event:
      print(event["contentBlockDelta"]["delta"]["text"], end="", flush=True)

### AWS Bedrock의 주요 기능 3가지 설명

AWS Bedrock는 AWS가 제공하는 지능형 장기 자원 관리 및 생성 AI 서비스입니다. 이 서비스는 여러 가지 주요 기능을 제공하며, 그 중 3가지를 설명해드리겠습니다.

---

#### **1. 다양한 LLM(대규모 언어 모델) 제공**

AWS Bedrock는 다양한 선도적인 대규모 언어 모델(LLM)을 한 자리에서 접근할 수 있게 합니다. 사용자는 **Amazon Bedrock 콘솔 또는 API**를 통해 **Anthropic의 Claude**, **Meta의 Llama**, **Stability AI의 StableLM**, **Amazon이 개발한 Titan 모델** 등을 바로 사용할 수 있습니다.

이 모델들은 각각 다른 강점을 가지고 있어, **문서 요약, 챗봇 구축, 콘텐츠 생성, 코드 작성** 등 다양한 작업에 활용할 수 있습니다.

> ✅ **장점**: 다양한 모델을 비교해 볼 수 있고, 하나의 통합된 플랫폼에서 관리할 수 있습니다.

---

#### **2. 안전한 AI 사용 및 관리 도구 제공**

AWS Bedrock는 AI 모델을 사용할 때 발생할 수 있는 **안전성, 책임성, 규정 준수** 문제를 해결하는 도구를 제공합니다. 주요 기능으로는 다음과 같습니다.

- **문서 분류 및 필터링**: 사용자 입력이 위험하거나 비윤리적인 내용을 포함하는지 검출합니다.
- **토큰화 및 내용 검토**: 생성된 출력물을 모니터링하고 관리할 수 있습니다.
- **사용자 정의 가이드라인 적용**: 특정 도메인이나 기업 정책에 맞게 모델의 출력물을 제한할 수 있습니다.

> ✅ **장점**: AI 시스템을 안전하게 운영하고, 규정 준수를 유지할 수 있습니다.

---

#### **3. 확장성과 통합성**

AWS Bedrock는 **AWS의 강력한 인프라와 통합성**을 제공하여 대규모 애플리케이션에 쉽게 적용할 수 있습니다.

- **AWS 서비스와의 통합**: Amazon S3, Dyn

## System Prompt & inferenceConfig

> 모델의 동작을 제어하는 핵심 파라미터입니다.  
> **System Prompt**로 모델의 역할과 규칙을 정의하고, **inferenceConfig**로 응답 생성 방식을 조절합니다.  
> 두 파라미터 모두 Converse API와 InvokeModel API에서 사용할 수 있습니다.

### System Prompt

> 모델에게 역할, 성격, 응답 규칙을 부여합니다.  
> `system` 파라미터에 리스트 형태로 전달합니다.

In [ ]:
# System Prompt 없이 호출
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "파이썬이 뭐야?"}]}]
)
print("[System Prompt 없음]")
print(response["output"]["message"]["content"][0]["text"])

In [ ]:
# System Prompt 적용 — 5살 아이에게 설명하는 선생님 역할
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  system=[{"text": "너는 5살 아이에게 설명하는 유치원 선생님이야. 쉬운 비유를 사용하고, 2문장 이내로 답해."}],
  messages=[{"role": "user", "content": [{"text": "파이썬이 뭐야?"}]}]
)
print("[System Prompt 적용]")
print(response["output"]["message"]["content"][0]["text"])

### inferenceConfig

> 모델의 응답 생성 방식을 제어하는 파라미터입니다.
>
> | 파라미터 | 설명 | 범위 |
> |----------|------|------|
> | `maxTokens` | 최대 응답 토큰 수 | 1 ~ 모델 최대값 |
> | `temperature` | 높을수록 창의적, 낮을수록 일관적 | 0.0 ~ 1.0 |
> | `topP` | 누적 확률 기준 후보 토큰 범위 | 0.0 ~ 1.0 |

#### maxTokens 비교
> 응답 길이를 제한합니다. 짧게 설정하면 응답이 중간에 잘립니다.

In [28]:
# maxTokens = 20 (짧게 제한)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS의 주요 서비스 5가지를 설명해줘"}]}],
  inferenceConfig={"maxTokens": 20}
)
print(f"[maxTokens=20] stopReason: {response['stopReason']}")
print(response["output"]["message"]["content"][0]["text"])

[maxTokens=20] stopReason: max_tokens
### AWS의 주요 서비스 5가지 설명

아마존 웹 서비스(AWS)는 클라


In [29]:
# maxTokens = 500 (충분히)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "AWS의 주요 서비스 5가지를 설명해줘"}]}],
  inferenceConfig={"maxTokens": 500}
)
print(f"[maxTokens=500] stopReason: {response['stopReason']}")
print(response["output"]["message"]["content"][0]["text"])

[maxTokens=500] stopReason: max_tokens
### AWS의 주요 서비스 5가지 설명

아마존 웹 서비스(AWS)는 클라우드 컴퓨팅 플랫폼으로, 다양한 서비스들을 제공합니다. 여기서 가장 대표적인 5가지 서비스를 설명해드리겠습니다.

---

#### 1. **Amazon EC2 (Elastic Compute Cloud)**
- **정의**: 가상화된 컴퓨팅 인스턴스를 제공하는 서비스입니다. 사용자는 필요에 따라 서버를 임대하여 운영할 수 있습니다.
- **주요 기능**
  - 다양한 인스턴스 유형 선택 (CPU, 메모리, 그래픽 처리 등)
  - 가용성 높은 아키텍처 (멀티-AZ 지원)
  - 보안 그룹 및 네트워크 제어
  - 인스턴스 스케일링 및 자동화 가능
- **사용 사례**
  - 웹 서버, 애플리케이션 서버 운영
  - 개발/테스트 환경 제공
  - 데이터 분석 및 배치 처리

---

#### 2. **Amazon S3 (Simple Storage Service)**
- **정의**: 오브젝트 저장소로, 무제한의 스케일링이 가능한 마사스 스토리지 서비스입니다.
- **주요 기능**
  - 안정적이고 저렴한 스토리지
  - 버킷 기반 관리
  - 버전 관리 및 복구 가능
  - 보안 설정 (IAM, 암호화 등)
  - 생명 주기 관리 (자동 삭제/이동)
- **사용 사례**
  - 파일 백업 및 아카이브
  - 웹사이트 호스팅
  - 데이터 호스팅 및 공유
  - AI/ML 모델 데이터 저장

---

#### 3. **Amazon RDS (Relational Database Service)**
- **정의**: 관계형 데이터베이스를 관리하기 위한 서비스로, 여러 데이터베이스 엔진을 지원합니다.
- **지원 엔진**
  - MySQL, PostgreSQL, Oracle, SQL Server, MariaDB, Aurora 등
- **주요 기능**
  - 자동 패치 및 백업
  - 스케일링 및 복구 용이
  - 다중 AZ 배포

#### temperature 비교
> 같은 질문을 temperature 0.0과 1.0으로 각각 호출하여 차이를 비교합니다.  
> - **0.0**: 가장 확률 높은 토큰만 선택 → 일관된 응답  
> - **1.0**: 다양한 토큰 선택 가능 → 창의적/랜덤한 응답
>
> 💡 여러 번 실행해보세요. temperature=0.0은 매번 거의 같은 답이 나오고, 1.0은 매번 다른 답이 나옵니다.

In [53]:
# temperature = 0.0 (결정적) — 여러 번 실행해도 같은 결과
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'여름'을 표현하는 독특한 단어나 표현 5개, 단어 및 표현만 출력해줘"}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 0.0}
)
print("[temperature=0.0]")
print(response["output"]["message"]["content"][0]["text"])

[temperature=0.0]
'여름'을 표현하는 독특한 단어와 표현 5개를 소개합니다:

1. **아지안**  
2. **한여름**  
3. **여름철**  
4. **여름바람**  
5. **여름비**


In [56]:
# temperature = 1.0 (창의적) — 실행할 때마다 다른 결과
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'여름'을 표현하는 독특한 단어나 표현 5개, 단어 및 표현만 출력해줘"}]}],
  inferenceConfig={"maxTokens": 200, "temperature": 1.0}
)
print("[temperature=1.0]")
print(response["output"]["message"]["content"][0]["text"])

[temperature=1.0]
'여름'을 표현하는 독특한 단어와 표현 5개를 소개합니다.

1. **아지랑이** - 여름에 마다 발생하는 갑작스러운 강한 바람을 비유적으로 일컫는 말.
2. **한여름** - 여름의 한가운데를 가리키는 표현.
3. **생추** - 여름에 먹는 시원한 음료와 관련된 표현으로, 생강을 넣은 따뜻한 물을 부어 마시는 전통 음료를 일컫는 말.
4. **개한여름** - 여름철에 태양이 가장 뜨거울 때를 비유적으로 이르는 표현.
5. **해거온도** - 여름철에 고온과 습도가 높은 날씨를 일컫는 표현으로, 사람이 불편함을 느끼는


#### topP
> 누적 확률 상위 P%의 토큰만 후보로 사용합니다.  
> - **0.1**: 상위 10% 토큰만 → 매우 보수적  
> - **0.9**: 상위 90% 토큰 → 다양한 표현 가능
>
> 💡 일반적으로 temperature와 topP 중 하나만 조절합니다.  
> 둘 다 높이면 너무 랜덤해지고, 둘 다 낮추면 반복적인 응답이 됩니다.  
> 실무에서는 temperature만 조절하는 경우가 많습니다.

In [79]:
# topP = 0.1 (보수적)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'바람이 ___' 시의 첫 줄을 완성해줘. 딱 그 줄만"}]}],
  inferenceConfig={"maxTokens": 200, "topP": 0.1}
)
print("[topP=0.1]")
print(response["output"]["message"]["content"][0]["text"])

[topP=0.1]
'바람이 **뜯어와**'  

이 구절은 한국 시인 김소월의 대표적인 시 '소나트'의 첫 줄입니다. 원문은 다음과 같습니다:

> **바람이 뜯어와 수풀을 흔들고**  
> **나의 마음을 뜯어와도 모르니**  
> **그 수풀은 아직도 내 가슴에 있다**

이 시는 자연의 풍경을 통해 내적 감정을 표현한 것으로 유명합니다.


In [81]:
# topP = 1.0 (다양한)
response = client.converse(
  modelId="us.amazon.nova-2-lite-v1:0",
  messages=[{"role": "user", "content": [{"text": "'바람이 ___' 시의 첫 줄을 완성해줘. 딱 그 줄만"}]}],
  inferenceConfig={"maxTokens": 200, "topP": 1.0}
)
print("[topP=1.0]")
print(response["output"]["message"]["content"][0]["text"])

[topP=1.0]
'바람이 **부는 곳으로 가고 싶다**'  

이 시는 한국의 유명한 시인 **김소월**이 쓴 시로, 그의 대표적 시집 《서정시집》에 수록된 시입니다. 첫 줄이 정확히 위와 같습니다.
